# Notebook 2 — Limpieza y Preprocesamiento
**Prerequisito:** Haber ejecutado Notebook 1 — EDA
**Objetivo:** Preparar los datos para modelamiento

## 1. Librerías e Ingesta

In [0]:
!pip install imbalanced-learn
from pyspark.sql import SparkSession
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

df_pd = spark.table("bank_additional_full").toPandas()
print(f"Shape original: {df_pd.shape}")



Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Shape original: (41188, 21)


## 2. Limpieza de Nombres de Columnas

In [0]:
# Los puntos en nombres de columnas generan conflictos en Spark y pandas
df_pd.columns = [c.replace(".", "_") for c in df_pd.columns]
print("Columnas limpias:")
print(df_pd.columns.tolist())

Columnas limpias:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'y']


## 3. Tratamiento de Valores Especiales

In [0]:
# pdays = 999 es un valor codificado que significa "nunca contactado"
# Se reemplaza por -1 para que los modelos lo traten como categoría especial
# (no usamos None/NaN porque SMOTE no acepta valores nulos)
df_pd["pdays"] = df_pd["pdays"].replace(999, -1)

print(f"Valores únicos en pdays (top 10): {sorted(df_pd['pdays'].unique())[:10]}")
print(f"Clientes nunca contactados: {(df_pd['pdays'] == -1).sum()}")

Valores únicos en pdays (top 10): [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
Clientes nunca contactados: 39673


## 4. Encoding de Variables Categóricas

In [0]:
# LabelEncoder convierte strings a números para que los modelos los procesen
cat_cols = ["job", "marital", "education", "contact",
            "month", "day_of_week", "poutcome",
            "default", "housing", "loan"]

le = LabelEncoder()
for c in cat_cols:
    df_pd[c] = le.fit_transform(df_pd[c].astype(str))

print("Encoding completado. Tipos de datos:")
print(df_pd.dtypes)

Encoding completado. Tipos de datos:
age                 int64
job                 int64
marital             int64
education           int64
default             int64
housing             int64
loan                int64
contact             int64
month               int64
day_of_week         int64
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome            int64
emp_var_rate      float64
cons_price_idx    float64
cons_conf_idx     float64
euribor3m         float64
nr_employed       float64
y                  object
dtype: object


## 5. Encoding de Variable Objetivo

In [0]:
# Convertir y: "yes" → 1, "no" → 0
df_pd["y"] = (df_pd["y"] == "yes").astype(int)
print(f"Distribución variable objetivo:\n{df_pd['y'].value_counts()}")

Distribución variable objetivo:
0    36548
1     4640
Name: y, dtype: int64


## 6. Eliminación de Variable con Data Leakage

In [0]:
# IMPORTANTE: 'duration' (duración de la llamada) solo se conoce DESPUÉS de la llamada,
# es decir, después de que el cliente ya tomó su decisión.
# Usarla para predecir sería trampa (data leakage) — el modelo no sería válido en producción.
# Referencia: documentación oficial del dataset UCI recomienda excluirla para modelos realistas.

X = df_pd.drop(["y", "duration"], axis=1)
y = df_pd["y"]

print(f"Features utilizadas ({len(X.columns)}):")
print(X.columns.tolist())

Features utilizadas (19):
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed']


## 7. División Train / Test

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]} registros")
print(f"Test:  {X_test.shape[0]} registros")
print(f"\nDistribución en train:\n{y_train.value_counts()}")
print(f"\nDistribución en test:\n{y_test.value_counts()}")

Train: 32950 registros
Test:  8238 registros

Distribución en train:
0    29238
1     3712
Name: y, dtype: int64

Distribución en test:
0    7310
1     928
Name: y, dtype: int64


## 8. Balanceo de Clases con SMOTE

In [0]:
# El dataset está desbalanceado (88.7% / 11.3%).
# SMOTE (Synthetic Minority Oversampling Technique) genera muestras sintéticas
# de la clase minoritaria interpolando entre ejemplos existentes.
# IMPORTANTE: SMOTE se aplica SOLO sobre train, nunca sobre test.

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"Antes del balanceo  — 0: {sum(y_train==0):,} | 1: {sum(y_train==1):,}")
print(f"Después del balanceo — 0: {sum(y_train_bal==0):,} | 1: {sum(y_train_bal==1):,}")

Exception ignored on calling ctypes callback function: <function _ThreadpoolInfo._find_modules_with_dl_iterate_phdr.<locals>.match_module_callback at 0xff0d40062520>
Traceback (most recent call last):
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 400, in match_module_callback
    self._make_module_from_path(filepath)
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 515, in _make_module_from_path
    module = module_class(filepath, prefix, user_api, internal_api)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 606, in __init__
    self.version = self.get_version()
                   ^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 646, in get_version
    config = get_config().split()
             ^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'split'


Antes del balanceo  — 0: 29,238 | 1: 3,712
Después del balanceo — 0: 29,238 | 1: 29,238


## 9. Feature Engineering

In [0]:
# Creación de variables derivadas con sentido de negocio
X_train_bal["fue_contactado"]    = (X_train_bal["pdays"] != -1).astype(int)
X_train_bal["contacto_intensivo"] = (X_train_bal["campaign"] > 3).astype(int)
X_train_bal["economia_favorable"] = (X_train_bal["emp_var_rate"] < 0).astype(int)

X_test["fue_contactado"]    = (X_test["pdays"] != -1).astype(int)
X_test["contacto_intensivo"] = (X_test["campaign"] > 3).astype(int)
X_test["economia_favorable"] = (X_test["emp_var_rate"] < 0).astype(int)

print(f"Features finales ({len(X_train_bal.columns)}):")
print(X_train_bal.columns.tolist())

Features finales (22):
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'fue_contactado', 'contacto_intensivo', 'economia_favorable']


## 10. Resumen del Preprocesamiento

| Paso | Acción | Motivo |
|---|---|---|
| Nombres columnas | Reemplazar puntos por _ | Compatibilidad Spark/pandas |
| pdays = 999 | Reemplazar por -1 | Valor codificado, no numérico real |
| Categóricas | LabelEncoder | Modelos requieren inputs numéricos |
| duration | Eliminada | Data leakage |
| Split | 80% train / 20% test | Evaluación honesta del modelo |
| SMOTE | Solo en train | Corregir desbalance sin contaminar test |
| Feature engineering | 3 variables nuevas | Capturar patrones de negocio |